# Training A Surrogate Model with PyTorch

This notebook illustrates how to train a simple neural network surrogate model using Pytorch. In this notebook we look at the simple case of the `arnett_with_features` function from redback. We show how to generate the training data, fit the model, and evaluate the model.

This notebook uses multiple packages that are not installed with redback_surrogates by default (and must be install manually), including:

   * redback (for the arnett model)
   * torch (for the training)

In [ ]:
import numpy as np

from astropy.table import Table
from redback.transient_models.supernova_models import arnett_with_features
from tqdm import tqdm
from scipy.stats import loguniform

from redback_surrogates.learned_surrogate_data import LearnedSurrogateDataset
from redback_surrogates.learned_surrogate_train import evaluate_learned_model, train_pytorch_model

## Creating the Training Data

We start with a simple method to create the training data. This consists of three steps:
   1) We generate all of the input parameters needed by the function.
   2) Generate an output grid (spectra) for each of the sets of parameters.
   3) Wrap the data in a `LearnedSurrogateDataset`.

We put these steps in a helper function so we can use the same approach for generating the test data.

In [ ]:
def generate_arnett_data(num_samples):
    """Generate training/validation data for the Arnett model with
    randomly sampled parameters and outputs as the spectra.

    :param num_samples: Number of samples to generate.
    """
    times = np.array([0.0, 1.0])  # Unused by "spectra" output type

    # Generate the parameter samples.
    data = {
        "redshift": loguniform(0.001, 3.0).rvs(size=num_samples),
        "f_nickel": loguniform(0.001, 1.0).rvs(size=num_samples),
        "mej": loguniform(0.0001, 100.0).rvs(size=num_samples),
        "vej": loguniform(1_000, 100_000).rvs(size=num_samples),
        "kappa": loguniform(0.0001, 10000.0).rvs(size=num_samples),
        "kappa_gamma": loguniform(0.01, 0.1).rvs(size=num_samples),
        "temperature_floor": loguniform(1_000, 100_000).rvs(size=num_samples),
    }
    
    # Compute the output for each sample.
    output_grids = []
    out_times = None
    out_lambdas = None
    for i in tqdm(range(num_samples)):
        grid = arnett_with_features(
            time=times,
            redshift=data["redshift"][i],
            f_nickel=data["f_nickel"][i],
            mej=data["mej"][i],
            vej=data["vej"][i],
            kappa=data["kappa"][i],
            kappa_gamma=data["kappa_gamma"][i],
            temperature_floor=data["temperature_floor"][i],
            output_format="spectra",
        )
        output_grids.append(grid.spectra)

        # Save the time and wavelength grid from the first sample.
        if i == 0:
            out_times = grid.time
            out_lambdas = grid.lambdas

    data["grid"] = np.array(output_grids)

    results = LearnedSurrogateDataset(
        Table(data),
        output_column="grid",
        times=out_times,
        wavelengths=out_lambdas,
    )
    return results

Now we create the actual training data with 2,000 samples.

In [ ]:
training_data = generate_arnett_data(num_samples=1_000)

We can look at the histogram of flux values to get an idea of what we are trying to predict.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)

allout = training_data.get_output()
allout[allout < 1e-30] = 1e-30  # Avoid log(0)

plt.hist(np.log10(allout.flatten()), bins=100, log=True)
plt.xlabel("log(Flux)")
plt.ylabel("Count")
plt.title("Distribution of Flux Values in Generated Data")
plt.show()

## Training the Model

Users can pick their favorite machine learning platform to train models. Here we use the included `train_pytorch_model` function, which trains a multi-layer neural network. The input layer will correspond to the parameters. The output layer will correspond to the SED grid as defined by the function (in this case 3000 time steps and 100 wavelengths). We use hidden layers of size 128, 64, and 128.

In [ ]:
model = train_pytorch_model(
    training_data,
    hidden_sizes=[32, 16, 64],
    training_epochs=50,
    verbose=True,
)

## Evaluating the Model

The returned object is a `LearnedSurrogateModel` that can be saved to disk as a ONNX model and used in redback. We can generate a new (random) test set and evaluate the performance of the model on that set.

In [ ]:
test_data = generate_arnett_data(num_samples=200)

mse, maxse = evaluate_learned_model(model, test_data)
print(f"Test MSE: {mse}")
print(f"Test Max SE: {maxse}")